In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

In [ ]:
FILE_PATH = "/content/mess_attendance_200_students_365_days.csv"

df = pd.read_csv("/content/mess_attendance_200_students_365_days.csv")

print("\nDataset loaded successfully!")
print("Shape:", df.shape)

print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

In [ ]:
df["date"] = pd.to_datetime(df["date"])

In [ ]:
df = df.sort_values("date").reset_index(drop=True)

In [ ]:
# Day number of month
df["day_of_month"] = df["date"].dt.day

# Month
df["month"] = df["date"].dt.month

# Day of year
df["day_of_year"] = df["date"].dt.dayofyear

# Week number
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

In [ ]:
TARGET = "lunch_attendance"
features = [
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "is_exam_period",
    "is_rainy",
    "is_special_event",
    "previous_day_lunch",
    "lunch_7_day_avg",
    "day_of_month",
    "month",
    "day_of_year",
    "week_of_year"
]
X = df[features]
y = df[TARGET]

In [ ]:
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
categorical_features = [
    "day_of_week"
]

numerical_features = [
    "is_weekend",
    "is_holiday",
    "is_exam_period",
    "is_rainy",
    "is_special_event",
    "previous_day_lunch",
    "lunch_7_day_avg",
    "day_of_month",
    "month",
    "day_of_year",
    "week_of_year"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),

        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

In [ ]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=10,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


print("\nTraining Random Forest...")

random_forest_model.fit(
    X_train,
    y_train
)

rf_predictions = random_forest_model.predict(
    X_test
)

In [ ]:
rf_mae = mean_absolute_error(
    y_test,
    rf_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_predictions
    )
)

rf_r2 = r2_score(
    y_test,
    rf_predictions
)

In [ ]:
print("\n==============================")
print("RANDOM FOREST RESULTS")
print("==============================")

print("MAE :", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R²  :", round(rf_r2, 3))

In [ ]:
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)


print("\nTraining Linear Regression...")

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

In [ ]:
linear_mae = mean_absolute_error(
    y_test,
    linear_predictions
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_predictions
    )
)

linear_r2 = r2_score(
    y_test,
    linear_predictions
)
print("\n==============================")
print("LINEAR REGRESSION RESULTS")
print("==============================")

print("MAE :", round(linear_mae, 2))
print("RMSE:", round(linear_rmse, 2))
print("R²  :", round(linear_r2, 3))

In [ ]:
comparison = pd.DataFrame({

    "Model": [
        "Linear Regression",
        "Random Forest"
    ],

    "MAE": [
        linear_mae,
        rf_mae
    ],

    "RMSE": [
        linear_rmse,
        rf_rmse
    ],

    "R2": [
        linear_r2,
        rf_r2
    ]
})

print("\n==============================")
print("MODEL COMPARISON")
print("==============================")

print(
    comparison.round(3)
)


In [ ]:
if rf_mae < linear_mae:

    best_model = random_forest_model
    best_predictions = rf_predictions
    best_model_name = "Random Forest"
    best_mae = rf_mae

else:

    best_model = linear_model
    best_predictions = linear_predictions
    best_model_name = "Linear Regression"
    best_mae = linear_mae


print("\n==============================")
print("BEST MODEL")
print("==============================")

print("Model:", best_model_name)
print("MAE:", round(best_mae, 2))

In [ ]:
results = pd.DataFrame({

    "date": df.iloc[split_index:]["date"].values,

    "actual": y_test.values,

    "predicted": best_predictions

})

results["error"] = (
    results["actual"] -
    results["predicted"]
)

results["absolute_error"] = (
    abs(results["error"])
)


print("\nPrediction results:")
print(results.head(20))

In [ ]:
plt.plot(
    results["date"],
    results["actual"],
    label="Actual Attendance"
)

plt.plot(
    results["date"],
    results["predicted"],
    label="Predicted Attendance"
)

plt.xlabel("Date")
plt.ylabel("Lunch Attendance")

plt.title(
    "Actual vs Predicted Lunch Attendance"
)

plt.legend()

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    results["error"],
    bins=15
)

plt.xlabel("Prediction Error")
plt.ylabel("Frequency")

plt.title(
    "Prediction Error Distribution"
)

plt.tight_layout()

plt.show()

In [ ]:
absolute_errors = results["absolute_error"]

mean_error = absolute_errors.mean()

std_error = absolute_errors.std()

print("\n==============================")
print("MODEL UNCERTAINTY")
print("==============================")

print(
    "Average absolute error:",
    round(mean_error, 2)
)

print(
    "Error standard deviation:",
    round(std_error, 2)
)

In [ ]:
def predict_attendance(
    date,
    is_holiday,
    is_exam_period,
    is_rainy,
    is_special_event,
    previous_day_lunch,
    lunch_7_day_avg
):

    date = pd.to_datetime(date)

    day_of_week = date.day_name()

    is_weekend = int(
        date.dayofweek >= 5
    )

    day_of_month = date.day

    month = date.month

    day_of_year = date.dayofyear

    week_of_year = date.isocalendar().week


    new_data = pd.DataFrame({

        "day_of_week": [
            day_of_week
        ],

        "is_weekend": [
            is_weekend
        ],

        "is_holiday": [
            is_holiday
        ],

        "is_exam_period": [
            is_exam_period
        ],

        "is_rainy": [
            is_rainy
        ],

        "is_special_event": [
            is_special_event
        ],

        "previous_day_lunch": [
            previous_day_lunch
        ],

        "lunch_7_day_avg": [
            lunch_7_day_avg
        ],

        "day_of_month": [
            day_of_month
        ],

        "month": [
            month
        ],

        "day_of_year": [
            day_of_year
        ],

        "week_of_year": [
            week_of_year
        ]
    })


    prediction = best_model.predict(
        new_data
    )[0]


    # Keep prediction between 0 and 200
    prediction = np.clip(
        prediction,
        0,
        200
    )


    # Approximate prediction range
    lower = max(
        0,
        prediction - mean_error
    )

    upper = min(
        200,
        prediction + mean_error
    )


    return prediction, lower, upper

In [ ]:
tomorrow = "2026-08-22"
prediction, lower, upper = predict_attendance(

    date=tomorrow,

    is_holiday=0,

    is_exam_period=0,

    is_rainy=0,

    is_special_event=0,

    previous_day_lunch=180,

    lunch_7_day_avg=175
)


print("\n==============================")
print("TOMORROW'S ATTENDANCE")
print("==============================")

print(
    "Predicted attendance:",
    round(prediction)
)

print(
    "Expected range:",
    round(lower),
    "-",
    round(upper)
)

In [ ]:
joblib.dump(
    best_model,
    "mess_attendance_model.pkl"
)

print(
    "\nModel saved as: mess_attendance_model.pkl"
)

In [ ]:
results.to_csv(
    "attendance_predictions.csv",
    index=False
)

print(
    "Prediction results saved as: attendance_predictions.csv"
)

print("\nDONE!")